# White House “Repair and Restoration”: a \$6M account that became a \$323M one

This notebook traces one Treasury account — **White House Repair and Restoration** (Executive Office of the President, account `011-X-0109`) — across FY2022–FY2026, straight from OMB's own files, **two independent ways**: the apportionment side (SF-132) and the execution side (SF-133).

It runs top to bottom against [data.blazingstaranalytics.com](https://data.blazingstaranalytics.com/)'s free, CC0 mirror of OMB SF-132 / SF-133 — no API key, no login. Every figure traces to an OMB source file that carries a verifiable SHA-256.

**The story:** the direct appropriation stayed flat at ~\$2.5M a year. The account's growth — from ~\$6M to \$323M — is almost entirely *reimbursable / collected* funds (Economy Act transfers and collections). Both sides of OMB's books agree.

In [ ]:
%pip install -q requests pandas openpyxl matplotlib

In [ ]:
import textwrap
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import requests

plt.rcParams["text.parse_math"] = False  # treat "$" as a literal dollar sign, not math mode

APPROP, REIMB, CARRY = "#69539E", "#2A9D8F", "#C9C9C9"  # appropriation / reimbursable / brought-forward
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "blazingstar-data-demo (https://github.com/abigailhaddad/blazingstar-data)"})


def get_json(url):
    r = SESSION.get(url, timeout=90); r.raise_for_status(); return r.json()


def get_bytes(url):
    r = SESSION.get(url, timeout=120); r.raise_for_status(); return r.content


def latest_schedule(fy, tafs):
    """Latest apportionment iteration's detailed schedule for a TAFS in a fiscal year."""
    idx = get_json(f"https://cdn.bzstr.co/sf132/fy{fy}/index.json")
    rows = [f for f in idx["files"] if f.get("file_type") == "data_json" and f["tafs"] == tafs]
    f = max(rows, key=lambda r: int(r["iteration"]))
    return get_json(f["mirror_url"])


def funding_buckets(js):
    """Split SF-132 schedule lines into appropriation / reimbursable / brought-forward ($ millions)."""
    approp = reimb = carry = 0
    for s in js["ScheduleData"]:
        ln = str(s["LineNumber"]); amt = s["ApprovedAmount"] or 0
        if ln.startswith("6") or ln in ("1910", "1920", "1930"):
            continue  # apportionment / total lines — skip to avoid double-counting
        if ln.startswith("11"):
            approp += amt          # direct appropriation
        elif ln.startswith(("17", "18")):
            reimb += amt           # spending authority from collections / reimbursements
        elif ln.startswith("10"):
            carry += amt           # unobligated balance brought forward / recoveries
    return approp / 1e6, reimb / 1e6, carry / 1e6

## 1. The funding history (apportionment side, SF-132)

For each fiscal year, take the latest apportionment iteration of the no-year account `011-X-0109` and split its budgetary resources by funding source.

In [ ]:
TAFS = "011-X-0109"
rows = []
for fy in range(2022, 2027):
    approp, reimb, carry = funding_buckets(latest_schedule(fy, TAFS))
    rows.append({"fiscal_year": fy, "appropriation": approp,
                 "reimbursable_collected": reimb, "brought_forward": carry,
                 "total": approp + reimb + carry})
history = pd.DataFrame(rows)
money = ["appropriation", "reimbursable_collected", "brought_forward", "total"]
history.style.format({c: "${:,.1f}M" for c in money}).hide(axis="index")

## 2. The chart

In [ ]:
fys = history["fiscal_year"].tolist()
ap = history["appropriation"].tolist()
rb = history["reimbursable_collected"].tolist()
cy = history["brought_forward"].tolist()
totals = history["total"].tolist()

fig, ax = plt.subplots(figsize=(10.5, 7.8))
fig.subplots_adjust(top=0.72, bottom=0.25, left=0.10, right=0.97)
x = range(len(fys))
ax.bar(x, cy, color=CARRY, label="Balance brought forward")
ax.bar(x, rb, bottom=cy, color=REIMB, label="Reimbursable / collected funds")
ax.bar(x, ap, bottom=[c + r for c, r in zip(cy, rb)], color=APPROP, label="Direct appropriation")

for xi, t in zip(x, totals):
    ax.text(xi, t + max(totals) * 0.015, f"${t:,.0f}M", ha="center", va="bottom",
            fontsize=11, fontweight="bold", color="#222")

ax.set_xticks(list(x)); ax.set_xticklabels([f"FY{f}" for f in fys], fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}M"))
ax.set_ylim(0, max(totals) * 1.06)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#ededed", lw=0.8); ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, fontsize=10.5)

last = len(fys) - 1
ax.annotate("Direct appropriation never moved:\n~$2.5M every year",
            xy=(last, totals[-1]), xytext=(last - 2.4, max(totals) * 0.86),
            fontsize=10.5, color=APPROP, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=APPROP, lw=1.4))
ax.annotate(f"Reimbursable / collected funds:\n${rb[0]:,.1f}M → ${rb[-1]:,.0f}M  ({rb[-1] / rb[0]:.0f}×)",
            xy=(last, cy[-1] + rb[-1] * 0.5), xytext=(last - 2.7, max(totals) * 0.56),
            fontsize=10.5, color=REIMB, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=REIMB, lw=1.4))

fig.text(0.10, 0.965, "White House Repair and Restoration",
         fontsize=21, fontweight="bold", color=APPROP, ha="left", va="top")
fig.text(0.10, 0.912,
         "Apportioned budget authority grew from ~$7M (FY2022) to $323M (FY2026), while the direct\n"
         "appropriation never moved off ~$2.5M.",
         fontsize=11.5, color="#333", ha="left", va="top", linespacing=1.5)
fig.text(0.10, 0.842,
         "“Reimbursable / collected funds” = money paid into the account from other agencies and outside\n"
         "sources (transfers, reimbursements, collections) — not dollars Congress appropriated.",
         fontsize=10, color=REIMB, ha="left", va="top", linespacing=1.5)

footnote = (
    "Account: Executive Office of the President, “Executive Residence at the White House,” Treasury account "
    "011-X-0109 (no-year). Bars show apportioned budgetary resources (the legal spending limit), latest OMB iteration "
    "each fiscal year — not cash outlays. “Reimbursable / collected funds” = SF-132 spending authority from "
    "anticipated collections, reimbursements & other; the multi-year sibling account is funded under the Economy Act "
    "(31 U.S.C. 1535). Two sibling accounts (annual FY2026, multi-year FY2025–29) bring the full FY2026 family to "
    "~$844M, of which ~$385M is anticipated (not-yet-realized) collections. Through May FY2026, ~$54M of ~$384M "
    "apportioned had been obligated. The records do not state the account’s purpose. Source: OMB SF-132, mirrored "
    "CC0 at data.blazingstaranalytics.com (each file carries a verifiable SHA-256); reproducible at "
    "github.com/abigailhaddad/blazingstar-data.")
fig.text(0.02, 0.175, textwrap.fill(footnote, 150), fontsize=7.3, color="#777",
         ha="left", va="top", linespacing=1.45)

fig.savefig("white_house_account.png", dpi=200, facecolor="white")
plt.show()

## 3. The fuller picture — the account family and what's been spent

The chart above is the single no-year account. The full FY2026 picture spans three Treasury accounts (different availability windows), and the execution data shows how little has actually been obligated so far.

In [ ]:
# The full FY2026 family — three Treasury accounts that share account number 0109
idx = get_json("https://cdn.bzstr.co/sf132/fy2026/index.json")
fam = pd.DataFrame([f for f in idx["files"] if f.get("file_type") == "data_json"
                    and f.get("agency") == "EOP" and str(f.get("tafs", "")).endswith("0109")])
fam["amount"] = pd.to_numeric(fam["total_approved_amount"], errors="coerce")
fam["iteration"] = pd.to_numeric(fam["iteration"], errors="coerce")
fam_latest = fam.sort_values("iteration").groupby("tafs", as_index=False).tail(1)
print("FY2026 apportioned budgetary resources, by Treasury account:")
for r in fam_latest.sort_values("amount", ascending=False).itertuples():
    print(f"  {r.tafs:20} ${r.amount / 1e6:>7,.0f}M   (approved {r.approval_timestamp[:10]})")
print(f"  {'FAMILY TOTAL':20} ${fam_latest['amount'].sum() / 1e6:>7,.0f}M")

# Execution: EOP SF-133, cumulative through the latest monthly report
sf = get_json("https://liatris.blazingstaranalytics.com/sf133/index.json")
eop = next(a for a in sf["agencies"] if a["human_name"] == "Executive Office of the President")
latest_file = eop["years"][0]["latest"]
raw = pd.ExcelFile(BytesIO(get_bytes(latest_file["url"]))).parse("Raw Data")
wh = raw[raw["TAFS"].astype(str).str.contains("0109", na=False)]
col = next(c for c in ["AMT_MAY", "AMT_APR", "AMT_FEB", "AMT_JAN", "AMT_NOV"]
           if c in raw.columns and raw[c].fillna(0).abs().sum() > 0)
piv = wh.pivot_table(index="TAFS", columns="LINENO", values=col, aggfunc="sum")
resources, obligated = piv[1910].sum(), piv[2190].sum()
print(f"\nExecution as of {latest_file['period_label']}: "
      f"${resources / 1e6:,.0f}M apportioned, ${obligated / 1e6:,.0f}M obligated "
      f"({obligated / resources:.0%}) — the rest apportioned but not yet committed.")

## 4. Cross-check: the execution side (SF-133)

Joe Carlile pointed out a second, independent way to the same picture: the SF-133's *budgetary-resources* lines (the 1000–1920 block) decompose the same total into funding sources, exactly like the SF-132 schedule. Two sides of OMB's books — and they agree.

Below, the same no-year account is rebuilt from the **execution** filings, FY2022–FY2026. The execution side even adds detail the apportionment side can't: how much has actually been *collected* vs. merely *anticipated*.

In [ ]:
MONTHS = ["OCT", "NOV", "DEC", "JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP"]


def sf133_funding(year_obj, tafs_contains=" /X"):
    """Funding-source split from an SF-133 file's budgetary-resources lines ($ millions)."""
    raw = pd.ExcelFile(BytesIO(get_bytes(year_obj["latest"]["url"]))).parse("Raw Data")
    cols = [f"AMT_{m}" for m in MONTHS if f"AMT_{m}" in raw.columns]
    col = [c for c in cols if raw[c].fillna(0).abs().sum() > 0][-1]   # latest populated period
    acct = raw[raw["TAFS"].astype(str).str.contains("0109", na=False)
               & raw["TAFS"].astype(str).str.contains(tafs_contains, na=False)]
    g = acct.groupby(acct["LINENO"].astype(str))[col].sum()
    L = lambda n: g.get(n, 0) / 1e6
    collected, anticipated = L("1700") + L("1800"), L("1740") + L("1840")
    return {"fiscal_year": int(year_obj["fiscal_year"]), "period": year_obj["latest"]["period_label"],
            "appropriation": L("1160") + L("1260"),          # appropriation (disc + mand)
            "collected": collected, "anticipated": anticipated,
            "reimbursable_collected": collected + anticipated,
            "brought_forward": L("1070"), "total": L("1910")}


history_sf133 = pd.DataFrame(sorted([sf133_funding(y) for y in eop["years"]],
                                    key=lambda d: d["fiscal_year"]))
cols = ["appropriation", "collected", "anticipated", "reimbursable_collected", "brought_forward", "total"]
history_sf133.style.format({c: "${:,.1f}M" for c in cols}).hide(axis="index")

## 5. Two ways, one answer

The apportionment side (SF-132) and the execution side (SF-133), the same account, side by side. The shape is identical and the totals match for every closed year — FY2026 differs only because the SF-133 is the May report, before the June apportionment actions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.6), sharey=True)
fig.subplots_adjust(top=0.73, bottom=0.10, left=0.07, right=0.97, wspace=0.08)

panels = [("Apportionment", "FY2026 — as of June 17", history),
          ("Execution", "FY2026 — through May (monthly)", history_sf133)]
for ax, (title, timing, df) in zip(axes, panels):
    fys = df["fiscal_year"].tolist()
    cy = df["brought_forward"].tolist()
    rb = df["reimbursable_collected"].tolist()
    ap = df["appropriation"].tolist()
    totals = df["total"].tolist()
    x = range(len(fys))
    ax.bar(x, cy, color=CARRY, label="Balance brought forward")
    ax.bar(x, rb, bottom=cy, color=REIMB, label="Reimbursable / collected funds")
    ax.bar(x, ap, bottom=[c + r for c, r in zip(cy, rb)], color=APPROP, label="Direct appropriation")
    for xi, t in zip(x, totals):
        ax.text(xi, t + max(totals) * 0.02, f"${t:,.0f}M", ha="center", va="bottom",
                fontsize=9.5, fontweight="bold", color="#222")
    ax.set_xticks(list(x)); ax.set_xticklabels([f"FY{f}" for f in fys], fontsize=10)
    ax.set_title(title, fontsize=14, fontweight="bold", color=APPROP, pad=20)
    ax.text(0.5, 1.012, timing, transform=ax.transAxes, ha="center", va="bottom",
            fontsize=10, color="#888", style="italic")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#ededed", lw=0.8); ax.set_axisbelow(True)

axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}M"))
axes[0].legend(loc="upper left", frameon=False, fontsize=9.5)
fig.suptitle("White House Repair and Restoration",
             fontsize=18, fontweight="bold", color=APPROP, y=0.965)
fig.text(0.5, 0.895,
         "Apportioned budget authority by funding source, confirmed two independent ways. The direct appropriation "
         "stayed flat\nat ~$2.5M; the growth is reimbursable / collected funds — money paid in from other agencies "
         "and outside sources.",
         ha="center", va="top", fontsize=10.5, color="#444", linespacing=1.5)
fig.text(0.07, 0.015, "Source: OMB SF-132 apportionments & SF-133 execution · data.blazingstaranalytics.com (CC0)",
         fontsize=8, color="#999", ha="left", va="bottom")
fig.savefig("white_house_two_ways.png", dpi=200, facecolor="white")
plt.show()

## A clean version for sharing

A stripped-down rendering — title and chart only, no footnote — for posting. Reuses the SF-132 `history` table from Section 1.

In [ ]:
lfys = history["fiscal_year"].tolist()
lap = history["appropriation"].tolist()
lrb = history["reimbursable_collected"].tolist()
lcy = history["brought_forward"].tolist()
ltot = history["total"].tolist()

fig, ax = plt.subplots(figsize=(10, 6.6))
fig.subplots_adjust(top=0.73, bottom=0.11, left=0.10, right=0.96)
x = range(len(lfys))
ax.bar(x, lcy, color=CARRY, label="Balance brought forward")
ax.bar(x, lrb, bottom=lcy, color=REIMB, label="Reimbursable / collected funds")
ax.bar(x, lap, bottom=[c + r for c, r in zip(lcy, lrb)], color=APPROP, label="Direct appropriation")
for xi, t in zip(x, ltot):
    ax.text(xi, t + max(ltot) * 0.015, f"${t:,.0f}M", ha="center", va="bottom",
            fontsize=11, fontweight="bold", color="#222")
ax.set_xticks(list(x)); ax.set_xticklabels([f"FY{f}" for f in lfys], fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}M"))
ax.set_ylim(0, max(ltot) * 1.06)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#ededed", lw=0.8); ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, fontsize=10.5)

last = len(lfys) - 1
ax.annotate("Direct appropriation never moved:\n~$2.5M every year",
            xy=(last, ltot[-1]), xytext=(last - 2.4, max(ltot) * 0.84),
            fontsize=10.5, color=APPROP, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=APPROP, lw=1.4))
ax.annotate(f"Reimbursable / collected funds:\n${lrb[0]:,.1f}M → ${lrb[-1]:,.0f}M  ({lrb[-1] / lrb[0]:.0f}×)",
            xy=(last, lcy[-1] + lrb[-1] * 0.5), xytext=(last - 2.7, max(ltot) * 0.52),
            fontsize=10.5, color=REIMB, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=REIMB, lw=1.4))

fig.text(0.10, 0.955, "White House Repair and Restoration",
         fontsize=21, fontweight="bold", color=APPROP, ha="left", va="top")
fig.text(0.10, 0.885,
         "Apportioned budget authority, FY2022–FY2026. The direct appropriation stayed flat at ~$2.5M; the growth is\n"
         "reimbursable / collected funds — money paid into the account from other agencies and outside sources.",
         fontsize=10.5, color="#444", ha="left", va="top", linespacing=1.5)
fig.text(0.10, 0.015, "Source: OMB SF-132 apportionments · data.blazingstaranalytics.com (CC0)",
         fontsize=8, color="#999", ha="left", va="bottom")
fig.savefig("white_house_linkedin.png", dpi=200, facecolor="white")
plt.show()

## Sources & caveats

- **Figures are apportioned budgetary resources** (the legal spending limit OMB sets), not cash outlays.
- **The charts are the no-year account** `011-X-0109`. The full FY2026 *family* spans three Treasury accounts and totals ~\$844M, of which ~\$385M is *anticipated* (not-yet-realized) collections — so lead with the per-account figures, not the family sum, when precision matters.
- **The two sides differ by timing, not substance.** SF-133 is cumulative through the latest monthly report (May), which predates the June 17 apportionment actions; that is the only reason the FY2026 totals differ.
- **The data does not state the account's purpose.** The schedules show reimbursable authority and nothing about what it funds.
- **Sources (all CC0):** OMB SF-132 apportionments and SF-133 execution, mirrored at [data.blazingstaranalytics.com](https://data.blazingstaranalytics.com/); FY2027 figures cross-checked against the President's Budget Appendix. Each apportionment file carries a verifiable SHA-256.